# 신용 데이터 기반 드론 배송 B2C 수요 분석

**목적:** 최근 1년간 성남시 전입 인구의 소득·소비 특성(신용3)과 전체 상주 성인 인구의 금융 행동(신용8)을 결합하여,  
드론 배송 B2C 서비스의 실질 수요 구역과 성장 잠재 구역을 구분하고 서비스 확장 우선순위를 제안합니다.

| 데이터 | 설명 | 공간 단위 |
|--------|------|-----------|
| 신용3 (GYEONGGI_IN_STAT) | 전입 인구 소득·소비·금융 특성 | 행정동 (BS_HCD) |
| 신용8 (GYEONGGI_DM_STAT) | 상주 성인 인구 금융 행동 | 법정동 (BCD) |

| 지수 구성 | 세부 | 가중치 |
|-----------|------|--------|
| 현재 상주 수요 점수 | 신용8 13개월 평균 | **0.5** |
| 유입 수요 점수 | 신용3 13개월 평균 | **0.3** |
| 성장 트렌드 점수 | 신용8·신용3 변화율 평균 | **0.2** |

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium import Choropleth
import seaborn as sns
import warnings
import glob
import os
import json

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────────────────────
NB_DIR    = os.path.dirname(os.path.abspath('__file__'))
BASE_DIR  = os.path.join(NB_DIR, '..')
DATA_DIR  = os.path.join(BASE_DIR, '..', 'Data')
PROC_DIR  = os.path.join(BASE_DIR, 'processed')

CRED3_GLOB     = os.path.join(DATA_DIR, '신용3', 'GYEONGGI_IN_STAT_*_PUBLIC_성남시.csv')
CRED8_GLOB     = os.path.join(DATA_DIR, '신용8', 'GYEONGGI_DM_STAT_*_성남시.csv')
BOUNDARY_PATH  = os.path.join(PROC_DIR, 'seongnam_boundary.gpkg')
CROSSWALK_PATH = os.path.join(PROC_DIR, 'admin_code_crosswalk.csv')
MAPPING_PATH   = os.path.join(DATA_DIR, '행정동_법정동_20260325.xlsx')  # 행안부 행정동-법정동 매핑

cred3_files = sorted(glob.glob(CRED3_GLOB))
cred8_files = sorted(glob.glob(CRED8_GLOB))

print(f'신용3 파일 수: {len(cred3_files)}개')
print(f'신용8 파일 수: {len(cred8_files)}개')
print(f'매핑 파일 존재: {os.path.exists(MAPPING_PATH)}')

# 성남시 시군구 코드
SEONGNAM_SIGUNGU = [41131, 41133, 41135]  # 수정구, 중원구, 분당구

---
## 1단계 — 데이터 전처리
### 1-1. 행정동 코드 교차 참조 테이블 로드

In [ ]:
# 행정동 코드 교차참조 (CSV_ADMI_CD ↔ DONG_NM ↔ GU_NM)
crosswalk = pd.read_csv(CROSSWALK_PATH, encoding='utf-8-sig')
crosswalk['CSV_ADMI_CD'] = crosswalk['CSV_ADMI_CD'].astype(int)

print('교차참조 테이블:')
display(crosswalk.head(10))
print(f'행정동 수: {len(crosswalk)}')

### 1-2. 법정동(BCD) → 행정동 매핑 테이블

> **출처:** 행정안전부 `행정동_법정동_20260325.xlsx` — 말소일자 없는 현행 유효 레코드만 사용합니다.  
> 1개 법정동이 복수 행정동에 걸치는 경우(예: 신흥동 법정 → 신흥1·2·3동) 모든 행정동에 동일하게 매핑합니다.  
> 신용8 집계 지표는 모두 **비율(per-capita)** 이므로 1:N 확장 시 분자·분모가 동일 배율로 늘어나 지수값은 보존됩니다.

In [ ]:
# 행안부 행정동-법정동 매핑 파일 로드
df_map_raw = pd.read_excel(MAPPING_PATH)

# 현재 유효 레코드: 말소일자 없음 + 성남시 + 읍면동명 있는 것(구/시 단위 제외)
df_map_sn = df_map_raw[
    df_map_raw['말소일자'].isna() &
    df_map_raw['시군구명'].str.contains('성남', na=False) &
    df_map_raw['읍면동명'].notna()
][['행정동코드', '읍면동명', '법정동코드']].copy()

# BCD(법정동코드 10자리) → admi_nm(읍면동명) 매핑 테이블 (1:N 포함)
bcd_mapping = (
    df_map_sn[['법정동코드', '읍면동명']]
    .drop_duplicates()
    .rename(columns={'법정동코드': 'BCD', '읍면동명': 'admi_nm'})
)
bcd_mapping['BCD'] = bcd_mapping['BCD'].astype(int)

print(f'매핑 파일 전체 레코드: {len(df_map_raw):,}개')
print(f'성남시 유효 매핑 레코드: {len(bcd_mapping)}개')
print(f'커버된 BCD(법정동) 수: {bcd_mapping["BCD"].nunique()}개')
print(f'커버된 행정동 수: {bcd_mapping["admi_nm"].nunique()}개')

# 1:N 매핑 현황 (하나의 법정동이 여러 행정동에 걸치는 경우)
bcd_n = bcd_mapping.groupby('BCD')['admi_nm'].apply(list).reset_index()
multi = bcd_n[bcd_n['admi_nm'].apply(len) > 1]
print(f'\n법정동 1:N 매핑 ({len(multi)}건):')
for _, row in multi.iterrows():
    print(f'  BCD {row["BCD"]}: {" · ".join(row["admi_nm"])}')

display(bcd_mapping.head(15))

### 1-3. 신용3 — 전입 인구 데이터 로딩 (13개월)

In [ ]:
frames3 = []
for fp in cred3_files:
    # 파일명에서 연월 추출: GYEONGGI_IN_STAT_2412_PUBLIC_성남시.csv → '202412'
    parts = os.path.basename(fp).split('_')
    ym_raw = parts[3]   # '2412' or '2501'
    ym = '20' + ym_raw if len(ym_raw) == 4 else ym_raw
    tmp = pd.read_csv(fp, encoding='utf-8-sig')
    tmp['month'] = ym
    frames3.append(tmp)

df3_raw = pd.concat(frames3, ignore_index=True)

# 성남시 전입 필터 (BS_SIGNGU = 수정구·중원구·분당구)
df3 = df3_raw[df3_raw['BS_SIGNGU'].isin(SEONGNAM_SIGUNGU)].copy()

# 행정동 코드 생성: BS_HCD(10자리) // 100 = CSV_ADMI_CD(8자리)
df3['CSV_ADMI_CD'] = (df3['BS_HCD'] // 100).astype(int)

# 행정동명 매핑
df3 = df3.merge(
    crosswalk[['CSV_ADMI_CD','DONG_NM','GU_NM']],
    on='CSV_ADMI_CD', how='left'
).rename(columns={'DONG_NM': 'admi_nm'})

print(f'신용3 전체  shape: {df3_raw.shape}')
print(f'성남시 필터 shape: {df3.shape}')
print(f'기간: {sorted(df3["month"].unique())}')
print(f'매핑 성공 행정동: {df3["admi_nm"].notna().sum()} / {len(df3)}')
display(df3.head(3))

### 1-4. 신용8 — 상주 인구 데이터 로딩 (13개월)

In [ ]:
frames8 = []
for fp in cred8_files:
    parts = os.path.basename(fp).split('_')
    ym_raw = parts[3]   # '2412'
    ym = '20' + ym_raw if len(ym_raw) == 4 else ym_raw
    tmp = pd.read_csv(fp, encoding='utf-8-sig')
    tmp['month'] = ym
    frames8.append(tmp)

df8_raw = pd.concat(frames8, ignore_index=True)

# 법정동 → 행정동 매핑 적용
df8 = df8_raw.merge(bcd_mapping, on='BCD', how='left')
unmapped = df8[df8['admi_nm'].isna()]['BCD'].unique()

print(f'신용8 shape: {df8_raw.shape}')
print(f'기간: {sorted(df8["month"].unique())}')
print(f'매핑 실패 BCD: {unmapped.tolist() if len(unmapped) > 0 else "없음"}')
print(f'AGE 범주: {sorted(df8["AGE"].unique())}')
display(df8.head(3))

---
## 2단계 — 지표 산출
### 2-1. 신용3 지표: 전입 인구 소득·소비 특성

In [ ]:
def weighted_avg(df, val_col, wt_col):
    """가중 평균 (총합 0인 경우 NaN)"""
    total_wt = df[wt_col].sum()
    if total_wt == 0:
        return np.nan
    return (df[val_col] * df[wt_col]).sum() / total_wt

# ── 월별·행정동별 집계 ──────────────────────────────────────────────────────
c3_records = []
for (month, dong), grp in df3.dropna(subset=['admi_nm']).groupby(['month', 'admi_nm']):
    tot = grp['TOT_CNT'].sum()
    if tot == 0:
        continue
    c3_records.append({
        'month'     : month,
        'admi_nm'   : dong,
        'tot_cnt'   : tot,
        'avg_inc'   : weighted_avg(grp, 'AVG_INC', 'TOT_CNT'),
        'avg_card'  : weighted_avg(grp, 'AVG_CARD', 'TOT_CNT'),
        'high_ratio': grp['HIGH_CNT'].sum() / tot,
        'avg_dist'  : weighted_avg(grp, 'AVG_DIST', 'TOT_CNT'),
    })

c3_monthly = pd.DataFrame(c3_records)
print(f'신용3 월별 집계 shape: {c3_monthly.shape}')
display(c3_monthly.sort_values(['admi_nm','month']).head(10))

In [ ]:
# ── 13개월 평균 지표 ────────────────────────────────────────────────────────
c3_avg = (
    c3_monthly
    .groupby('admi_nm')[['avg_inc','avg_card','high_ratio','avg_dist']]
    .mean()
    .reset_index()
)

# ── 변화율: 202412 → 202512 ─────────────────────────────────────────────────
def calc_change_rate(df, col, start='202412', end='202512'):
    s = df[df['month'] == start].set_index('admi_nm')[col].rename('start')
    e = df[df['month'] == end  ].set_index('admi_nm')[col].rename('end')
    merged = pd.concat([s, e], axis=1)
    return ((merged['end'] - merged['start']) / merged['start'].abs().replace(0, np.nan)).rename(col + '_chg')

c3_inc_chg  = calc_change_rate(c3_monthly, 'avg_inc').reset_index()
c3_card_chg = calc_change_rate(c3_monthly, 'avg_card').reset_index()

c3_idx = (
    c3_avg
    .merge(c3_inc_chg,  on='admi_nm', how='left')
    .merge(c3_card_chg, on='admi_nm', how='left')
    .fillna(0)
)

print('신용3 지표 테이블:')
display(c3_idx.sort_values('avg_inc', ascending=False).head(10))

### 2-2. 신용8 지표: 상주 인구 금융 행동

In [ ]:
# ── 월별·행정동별 집계 ──────────────────────────────────────────────────────
c8_records = []
for (month, dong), grp in df8.dropna(subset=['admi_nm']).groupby(['month', 'admi_nm']):
    tot = grp['TOT_CNT'].sum()
    if tot == 0:
        continue
    # 30~40대 인구
    age_3040 = grp[grp['AGE'].isin([30, 40])]['TOT_CNT'].sum()

    c8_records.append({
        'month'          : month,
        'admi_nm'        : dong,
        'tot_cnt'        : tot,
        'inc_per_cap'    : grp['SUM_INC'].sum()  / tot,   # 1인당 추정 월소득
        'card_per_cap'   : grp['SUM_CARD'].sum() / tot,   # 1인당 카드 이용금액
        'age3040_ratio'  : age_3040 / tot,                # 30~40대 비중
        'econ_ratio'     : grp['ECON_CNT'].sum() / tot,   # 경제활동 인구 비율
        'stay2yr_ratio'  : grp['STAY_2YRS_CNT'].sum() / tot,  # 2년 이상 정주 비율
    })

c8_monthly = pd.DataFrame(c8_records)
print(f'신용8 월별 집계 shape: {c8_monthly.shape}')
display(c8_monthly.sort_values(['admi_nm','month']).head(10))

In [ ]:
# ── 13개월 평균 지표 ────────────────────────────────────────────────────────
c8_avg = (
    c8_monthly
    .groupby('admi_nm')[['inc_per_cap','card_per_cap','age3040_ratio','econ_ratio','stay2yr_ratio']]
    .mean()
    .reset_index()
)

# ── 변화율 ─────────────────────────────────────────────────────────────────
c8_inc_chg  = calc_change_rate(c8_monthly, 'inc_per_cap').reset_index()
c8_card_chg = calc_change_rate(c8_monthly, 'card_per_cap').reset_index()

c8_idx = (
    c8_avg
    .merge(c8_inc_chg,  on='admi_nm', how='left')
    .merge(c8_card_chg, on='admi_nm', how='left')
    .fillna(0)
)

print('신용8 지표 테이블:')
display(c8_idx.sort_values('inc_per_cap', ascending=False).head(10))

---
## 3단계 — 드론 배송 수요 잠재 지수 산출

| 구성 | 세부 지표 | 가중치 |
|------|-----------|--------|
| **현재 상주 수요** (신용8) | 1인당 소득(0.3) · 1인당 카드(0.3) · 30-40대 비중(0.2) · 경제활동 비율(0.1) · 정주 비율(0.1) | **0.5** |
| **유입 수요** (신용3) | 전입자 소득(0.3) · 전입자 카드(0.3) · 고소득 비율(0.2) · 원거리 출퇴근(0.2) | **0.3** |
| **성장 트렌드** (변화율) | 신용8 소득 변화(0.4) · 신용8 카드 변화(0.3) · 신용3 소득 변화(0.2) · 신용3 카드 변화(0.1) | **0.2** |

In [ ]:
def minmax(s: pd.Series) -> pd.Series:
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

# 전체 행정동 기준표 (신용8 커버 행정동 + 신용3 커버 행정동 합집합)
all_dongs = sorted(
    set(c8_idx['admi_nm']) | set(c3_idx['admi_nm'])
)
base = pd.DataFrame({'admi_nm': all_dongs})

df_all = (
    base
    .merge(c8_idx, on='admi_nm', how='left')
    .merge(c3_idx, on='admi_nm', how='left', suffixes=('_c8','_c3'))
    .fillna(0)
)

# ── 현재 상주 수요 점수 (신용8) ─────────────────────────────────────────────
df_all['n_inc_pc']      = minmax(df_all['inc_per_cap'])
df_all['n_card_pc']     = minmax(df_all['card_per_cap'])
df_all['n_age3040']     = minmax(df_all['age3040_ratio'])
df_all['n_econ']        = minmax(df_all['econ_ratio'])
df_all['n_stay2yr']     = minmax(df_all['stay2yr_ratio'])

df_all['resident_score'] = (
    df_all['n_inc_pc']  * 0.3 +
    df_all['n_card_pc'] * 0.3 +
    df_all['n_age3040'] * 0.2 +
    df_all['n_econ']    * 0.1 +
    df_all['n_stay2yr'] * 0.1
)

# ── 유입 수요 점수 (신용3) ──────────────────────────────────────────────────
df_all['n_avg_inc']  = minmax(df_all['avg_inc'])
df_all['n_avg_card'] = minmax(df_all['avg_card'])
df_all['n_high']     = minmax(df_all['high_ratio'])
df_all['n_dist']     = minmax(df_all['avg_dist'])

df_all['inflow_score'] = (
    df_all['n_avg_inc']  * 0.3 +
    df_all['n_avg_card'] * 0.3 +
    df_all['n_high']     * 0.2 +
    df_all['n_dist']     * 0.2
)

# ── 성장 트렌드 점수 (변화율) ───────────────────────────────────────────────
# 극단값 클리핑 (95th percentile)
for col in ['inc_per_cap_chg','card_per_cap_chg','avg_inc_chg','avg_card_chg']:
    cap_hi = df_all[col].quantile(0.95)
    cap_lo = df_all[col].quantile(0.05)
    df_all[col + '_clip'] = df_all[col].clip(lower=cap_lo, upper=cap_hi)

df_all['n_c8_inc_chg']  = minmax(df_all['inc_per_cap_chg_clip'])
df_all['n_c8_card_chg'] = minmax(df_all['card_per_cap_chg_clip'])
df_all['n_c3_inc_chg']  = minmax(df_all['avg_inc_chg_clip'])
df_all['n_c3_card_chg'] = minmax(df_all['avg_card_chg_clip'])

df_all['trend_score'] = (
    df_all['n_c8_inc_chg']  * 0.4 +
    df_all['n_c8_card_chg'] * 0.3 +
    df_all['n_c3_inc_chg']  * 0.2 +
    df_all['n_c3_card_chg'] * 0.1
)

# ── 최종 드론 배송 수요 잠재 지수 ──────────────────────────────────────────
df_all['demand_index'] = (
    df_all['resident_score'] * 0.5 +
    df_all['inflow_score']   * 0.3 +
    df_all['trend_score']    * 0.2
)
df_all['demand_index_100'] = (df_all['demand_index'] * 100).round(2)

result = df_all[[
    'admi_nm','resident_score','inflow_score','trend_score','demand_index_100'
]].sort_values('demand_index_100', ascending=False)

print('=== B2C 드론 배송 수요 잠재 지수 TOP 20 ===')
display(result.head(20))

---
## 4단계 — 수요 유형 분류 및 시각화
### 4-1. 4분면 분류

In [ ]:
# 현재 수요 = resident_score + inflow_score 평균
# 성장 트렌드 = trend_score
df_all['current_demand'] = (df_all['resident_score'] * 0.5 + df_all['inflow_score'] * 0.3) / 0.8

demand_med  = df_all['current_demand'].median()
trend_med   = df_all['trend_score'].median()

def classify_quadrant(row):
    hi_d = row['current_demand'] >= demand_med
    hi_t = row['trend_score']    >= trend_med
    if hi_d and hi_t:   return '즉시 확장 구역'
    if hi_d and not hi_t: return '안정 수요 구역'
    if not hi_d and hi_t: return '성장 잠재 구역'
    return '관망 구역'

df_all['zone_type'] = df_all.apply(classify_quadrant, axis=1)

zone_counts = df_all['zone_type'].value_counts()
print('수요 유형 분포:')
for zone, cnt in zone_counts.items():
    print(f'  {zone}: {cnt}개 행정동')

print()
print('즉시 확장 구역:')
display(df_all[df_all['zone_type']=='즉시 확장 구역'][['admi_nm','demand_index_100','current_demand','trend_score']]
        .sort_values('demand_index_100', ascending=False))

### 4-2. 수요 잠재 지수 막대 차트

In [ ]:
ZONE_COLORS = {
    '즉시 확장 구역' : '#e63946',
    '안정 수요 구역' : '#457b9d',
    '성장 잠재 구역' : '#2a9d8f',
    '관망 구역'      : '#adb5bd',
}

top_result = df_all.sort_values('demand_index_100', ascending=False).head(20)

fig1 = px.bar(
    top_result.sort_values('demand_index_100'),
    x='demand_index_100',
    y='admi_nm',
    orientation='h',
    color='zone_type',
    color_discrete_map=ZONE_COLORS,
    text='demand_index_100',
    title='행정동별 B2C 드론 배송 수요 잠재 지수 TOP 20',
    labels={
        'demand_index_100': '수요 잠재 지수 (0~100)',
        'admi_nm': '행정동',
        'zone_type': '수요 유형'
    }
)
fig1.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig1.update_layout(
    height=620,
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig1.show()

### 4-3. 4분면 산점도

In [ ]:
fig2 = px.scatter(
    df_all,
    x='current_demand',
    y='trend_score',
    color='zone_type',
    color_discrete_map=ZONE_COLORS,
    size='demand_index_100',
    size_max=30,
    text='admi_nm',
    hover_data=['demand_index_100','resident_score','inflow_score','trend_score'],
    title='B2C 드론 배송 수요 4분면 매트릭스',
    labels={
        'current_demand': '현재 수요 점수 (상주+유입)',
        'trend_score'   : '성장 트렌드 점수',
        'zone_type'     : '수요 유형'
    }
)
fig2.update_traces(textposition='top center', textfont_size=9)

# 사분면 기준선
fig2.add_vline(x=demand_med, line_dash='dash', line_color='gray', opacity=0.6)
fig2.add_hline(y=trend_med,  line_dash='dash', line_color='gray', opacity=0.6)

# 사분면 레이블
x_max = df_all['current_demand'].max()
y_max = df_all['trend_score'].max()
annotations = [
    dict(x=x_max*0.9, y=y_max*0.95, text='즉시 확장', showarrow=False,
         font=dict(color='#e63946', size=11)),
    dict(x=x_max*0.9, y=y_max*0.05, text='안정 수요', showarrow=False,
         font=dict(color='#457b9d', size=11)),
    dict(x=x_max*0.05, y=y_max*0.95, text='성장 잠재', showarrow=False,
         font=dict(color='#2a9d8f', size=11)),
    dict(x=x_max*0.05, y=y_max*0.05, text='관망', showarrow=False,
         font=dict(color='gray', size=11)),
]
fig2.update_layout(
    height=600,
    plot_bgcolor='white',
    annotations=annotations,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig2.show()

### 4-4. 월별 소득·소비 추이 (상위 행정동)

In [ ]:
# 즉시 확장 구역 + 성장 잠재 구역 상위 동 선택
priority_zones = ['즉시 확장 구역', '성장 잠재 구역']
top_dongs = (
    df_all[df_all['zone_type'].isin(priority_zones)]
    .sort_values('demand_index_100', ascending=False)
    ['admi_nm'].tolist()[:8]
)

# 신용8 월별 소득 추이
c8_trend = c8_monthly[c8_monthly['admi_nm'].isin(top_dongs)].copy()
c8_trend['date']  = pd.to_datetime(c8_trend['month'], format='%Y%m')
c8_trend['label'] = c8_trend['date'].dt.strftime('%y.%m')
c8_trend = c8_trend.sort_values('date')

# 전체 월 × 동 조합 완성
all_months = sorted(c8_monthly['month'].unique())
full_idx = pd.MultiIndex.from_product([all_months, top_dongs], names=['month','admi_nm'])
c8_full = (
    c8_trend.set_index(['month','admi_nm'])[['inc_per_cap','card_per_cap']]
    .reindex(full_idx)
    .reset_index()
)
c8_full['date']  = pd.to_datetime(c8_full['month'], format='%Y%m')
c8_full['label'] = c8_full['date'].dt.strftime('%y.%m')
c8_full = c8_full.sort_values('date')

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['월별 1인당 추정 소득 (만원)', '월별 1인당 카드 이용금액 (만원)']
)

colors_palette = px.colors.qualitative.Set2

for i, dong in enumerate(top_dongs):
    sub = c8_full[c8_full['admi_nm'] == dong].sort_values('date')
    color = colors_palette[i % len(colors_palette)]
    zone = df_all[df_all['admi_nm'] == dong]['zone_type'].values[0] if len(df_all[df_all['admi_nm'] == dong]) else ''
    name = f"{dong} ({zone[:2]})" if zone else dong

    fig3.add_trace(go.Scatter(
        x=sub['label'], y=sub['inc_per_cap'] / 10000,
        mode='lines+markers', name=name, line=dict(color=color),
        legendgroup=dong, showlegend=True
    ), row=1, col=1)

    fig3.add_trace(go.Scatter(
        x=sub['label'], y=sub['card_per_cap'] / 10000,
        mode='lines+markers', name=name, line=dict(color=color),
        legendgroup=dong, showlegend=False
    ), row=1, col=2)

fig3.update_layout(
    title_text='주요 행정동 월별 소득·소비 추이 (신용8 기반)',
    height=450,
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.25)
)
fig3.show()

In [ ]:
# 신용3 전입자 월별 소득 추이 (성장 잠재 구역 중심)
c3_trend_top = c3_monthly[c3_monthly['admi_nm'].isin(top_dongs)].copy()
c3_trend_top['date']  = pd.to_datetime(c3_trend_top['month'], format='%Y%m')
c3_trend_top['label'] = c3_trend_top['date'].dt.strftime('%y.%m')

fig4 = px.line(
    c3_trend_top.sort_values('date'),
    x='label', y='avg_inc',
    color='admi_nm',
    markers=True,
    title='주요 행정동 전입자 평균 소득 월별 추이 (신용3 기반)',
    labels={'label': '연월', 'avg_inc': '전입자 평균 소득 (만원)', 'admi_nm': '행정동'}
)
fig4.update_layout(
    height=420,
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig4.show()

### 4-5. 단계구분도 (Choropleth Map)

In [ ]:
# 경계 파일 로드
gdf = gpd.read_file(BOUNDARY_PATH, layer='dong')
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# 지수 합류
gdf_m = gdf.merge(
    df_all[['admi_nm','demand_index_100','zone_type','resident_score','inflow_score','trend_score']],
    left_on='ADM_NM', right_on='admi_nm', how='left'
)

print(f'매칭: {gdf_m["demand_index_100"].notna().sum()} / {len(gdf_m)}')
print(f'미매칭: {gdf_m[gdf_m["demand_index_100"].isna()]["ADM_NM"].tolist()}')

In [ ]:
geojson_data = json.loads(gdf_m.to_json())
center = [gdf_m.geometry.centroid.y.mean(), gdf_m.geometry.centroid.x.mean()]

m = folium.Map(location=center, zoom_start=12, tiles='CartoDB positron')

# 수요 잠재 지수 단계구분도
Choropleth(
    geo_data=geojson_data,
    data=gdf_m.dropna(subset=['demand_index_100']),
    columns=['ADM_NM', 'demand_index_100'],
    key_on='feature.properties.ADM_NM',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.5,
    legend_name='B2C 드론 배송 수요 잠재 지수 (0~100)',
    nan_fill_color='#cccccc',
    nan_fill_opacity=0.3
).add_to(m)

# 툴팁
folium.GeoJson(
    geojson_data,
    tooltip=folium.GeoJsonTooltip(
        fields=['ADM_NM','demand_index_100','zone_type','resident_score','trend_score'],
        aliases=['행정동','수요 지수','수요 유형','상주 수요','성장 트렌드'],
        localize=True,
        sticky=True
    ),
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0}
).add_to(m)

# 수요 유형별 마커 (즉시 확장·성장 잠재 구역)
ZONE_ICON_COLORS = {
    '즉시 확장 구역' : 'red',
    '성장 잠재 구역' : 'green',
    '안정 수요 구역' : 'blue',
    '관망 구역'      : 'gray',
}
for _, row in gdf_m[gdf_m['zone_type'].isin(['즉시 확장 구역','성장 잠재 구역'])].iterrows():
    if row.geometry is None:
        continue
    centroid = row.geometry.centroid
    folium.Marker(
        location=[centroid.y, centroid.x],
        popup=folium.Popup(
            f"<b>{row['ADM_NM']}</b><br>"
            f"유형: {row['zone_type']}<br>"
            f"지수: {row['demand_index_100']:.1f}<br>"
            f"상주 수요: {row['resident_score']:.3f}<br>"
            f"성장 트렌드: {row['trend_score']:.3f}",
            max_width=220
        ),
        icon=folium.Icon(
            color=ZONE_ICON_COLORS.get(row['zone_type'], 'gray'),
            icon='home', prefix='fa'
        )
    ).add_to(m)

folium.LayerControl().add_to(m)

map_path = os.path.join(PROC_DIR, 'b2c_demand_index_map.html')
m.save(map_path)
print(f'지도 저장: {map_path}')
m

### 4-6. 지수 구성 히트맵

In [ ]:
hm_cols = {
    'n_inc_pc'   : '1인당 소득',
    'n_card_pc'  : '1인당 카드',
    'n_age3040'  : '30-40대 비중',
    'n_econ'     : '경제활동 비율',
    'n_stay2yr'  : '정주 비율',
    'n_avg_inc'  : '전입자 소득',
    'n_avg_card' : '전입자 카드',
    'n_high'     : '고소득 비율',
    'n_c8_inc_chg' : '소득 성장률',
    'n_c8_card_chg': '소비 성장률',
}

hm_data = (
    df_all
    .sort_values('demand_index_100', ascending=False)
    .head(20)
    .set_index('admi_nm')[list(hm_cols.keys())]
    .rename(columns=hm_cols)
)

fig_hm, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    hm_data,
    cmap='RdYlGn',
    linewidths=0.5,
    annot=True,
    fmt='.2f',
    ax=ax,
    vmin=0, vmax=1,
    cbar_kws={'label': '정규화 지수 (0~1)'}
)
ax.set_title('행정동별 B2C 드론 수요 세부 지표 히트맵 (TOP 20)', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
hm_path = os.path.join(PROC_DIR, 'b2c_heatmap.png')
plt.savefig(hm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'히트맵 저장: {hm_path}')

---
## 결과 저장 및 요약

In [ ]:
# 최종 결과 저장
out_cols = [
    'admi_nm','zone_type','demand_index_100',
    'resident_score','inflow_score','trend_score',
    'inc_per_cap','card_per_cap','age3040_ratio','econ_ratio','stay2yr_ratio',
    'avg_inc','avg_card','high_ratio','avg_dist',
    'inc_per_cap_chg','card_per_cap_chg','avg_inc_chg','avg_card_chg'
]
export_df = df_all[[c for c in out_cols if c in df_all.columns]].sort_values('demand_index_100', ascending=False)

result_path = os.path.join(PROC_DIR, 'b2c_demand_index.csv')
export_df.to_csv(result_path, index=False, encoding='utf-8-sig')
print(f'저장: {result_path}')

# 월별 신용8 집계 저장
c8_monthly.to_csv(os.path.join(PROC_DIR, 'b2c_monthly_c8.csv'), index=False, encoding='utf-8-sig')
c3_monthly.to_csv(os.path.join(PROC_DIR, 'b2c_monthly_c3.csv'), index=False, encoding='utf-8-sig')

print()
print('=== 최종 결과 요약 ===')
print(f'분석 행정동 수: {len(export_df)}개')
for zone in ['즉시 확장 구역','성장 잠재 구역','안정 수요 구역','관망 구역']:
    dongs = export_df[export_df['zone_type']==zone]['admi_nm'].tolist()
    print(f'\n[{zone}] ({len(dongs)}개)')
    for d in dongs:
        idx = export_df[export_df['admi_nm']==d]['demand_index_100'].values[0]
        print(f'  - {d}: {idx:.1f}')

---
## 한계 및 유의사항

1. **13개월 트렌드의 한계** — 단기 변동(계절성, 일회성 이슈)과 구조적 트렌드를 완전히 구분하기 어렵습니다. 연간 계절성 분석에는 충분하나, 장기 구조 변화 해석은 제한적입니다.

2. **법정동-행정동 1:N 매핑** — 신용8 BCD(법정동)는 행정동과 1:1 대응되지 않으며, 1개 법정동이 복수 행정동에 걸칩니다(예: 신흥동 법정 → 신흥1·2·3동). 본 분석에서는 행정안전부 `행정동_법정동_20260325.xlsx` 기준으로 1:N 확장 매핑을 적용하였습니다. 집계 지표가 모두 비율(per-capita)이므로 지수 산출에는 영향이 없으나, 법정동 내 세부 행정동 간 실제 분포 차이는 반영되지 않습니다.

3. **신용8 모집단 편향** — KCB(한국신용정보원) 등록 성인 기준 집계로 금융 이력이 없는 인구(외국인, 일부 청년층 등)는 과소 대표될 수 있습니다.

4. **신용3 전입 기간 편향** — 분석 기간(202412~202512) 내 전입한 인구만 반영하므로 장기 거주자의 소득·소비 패턴은 신용8에서만 확인 가능합니다.

5. **데이터 커버리지 차이** — 신용8은 43개 법정동 커버, 신용3은 50개 행정동 커버로, 신용8 미커버 행정동의 상주 수요 점수는 0으로 처리되어 과소평가될 수 있습니다.